In [ ]:
# %% Minimal setup from class

import os, json, textwrap, re, time
import requests

API_KEY  = os.getenv("OPENAI_API_KEY", "cse476")
API_BASE = os.getenv("API_BASE", "http://10.4.58.53:41701/v1")  
MODEL    = os.getenv("MODEL_NAME", "Milkbot")              

SYSTEM_PROMPT = "You are a helpful assistant. Reply with only the final answer—no explanation.",
TEMPERATURE   = 0.0 #Must be a float

def call_model_chat_completions(prompt: str,
                                system: str = SYSTEM_PROMPT,
                                model: str = MODEL,
                                temperature: float = TEMPERATURE,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 128,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}

def self_evaluate(question, prediction, expected_answer, model=MODEL):
    """
    Use the model itself as a strict grader.
    Returns True if the model says the prediction matches the expected answer; else False.
    Falls back to a simple normalized string compare if the model's reply is malformed.
    """
    import re

    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."
    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise, return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False
"""

    r = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (r.get("text") or "").strip().lower()
    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: simple normalization-based equality
    norm = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return norm(prediction) == norm(expected_answer)

def self_evaluate_tests(tests, model=MODEL, grader_model=None, sleep_sec=0.2, verbose=True):
    """
    Run the tests by querying the model for each prompt, then use LLM-as-a-judge
    (self_evaluate) to determine correctness.

    Args:
        tests: list of dicts with keys: id, prompt, expected (and optionally type)
        model: model used to generate predictions
        grader_model: model used to judge correctness (defaults to `model` if None)
        sleep_sec: small delay between calls to be polite to the API
        verbose: if True, print a summary line per test

    Returns:
        rows: list of dicts with fields:
              id, expected, got, correct, status, error
    """
    import time

    judge_model = grader_model or model
    rows = []

    for t in tests:
        # 1) Get model prediction
        r = call_model_chat_completions(
            t["prompt"],
            system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
            model=model,
            temperature=0.0,
        )
        got = (r.get("text") or "").strip()

        # 2) LLM-as-a-judge: strict True/False
        is_correct = self_evaluate(
            question=t["prompt"],
            prediction=got,
            expected_answer=t["expected"],
            model=judge_model,
        )

        row = {
            "id": t.get("id", "<unnamed>"),
            "expected": t["expected"],
            "got": got,
            "correct": bool(is_correct),
            "status": r.get("status"),
            "error": r.get("error"),
        }
        rows.append(row)

        if verbose:
            mark = "✅" if is_correct else "❌"
            print(f"{mark} {row['id']}: expected={row['expected']!r}, got={row['got']!r} (HTTP {row['status']})")
            if row["error"]:
                print("   error:", row["error"])

        if sleep_sec:
            time.sleep(sleep_sec)

    return rows


In [ ]:
# %% Define three tests: input + expected
tests = [
    {
        "id": "logic_race",
        "prompt": "Let $ABCD$ be a convex quadrilateral with $AB = CD = 10$ , $BC = 14$ , and $AD = 2\\sqrt{65}$ . Assume that the diagonals of $ABCD$ intersect at point $P$ , and that the sum of the areas of triangles $APB$ and $CPD$ equals the sum of the areas of triangles $BPC$ and $APD$ . Find the area of quadrilateral $ABCD$ .",
        "expected": "112",
        "type": "math"
    },
    {
        "id": "logic_race",
        "prompt": "A tennis player computes her win ratio by dividing the number of matches she has won by the total number of matches she has played. At the start of a weekend, her win ratio is exactly $0.500$ . During the weekend, she plays four matches, winning three and losing one. At the end of the weekend, her win ratio is greater than $.503$ . What's the largest number of matches she could've won before the weekend began?",
        "expected": "164",
        "type": "math"
    },
    {
        "id": "logic_race",
        "id": "math_inequality",
        "type": "numeric",  # grader will prefer numeric extraction
        "prompt": "Solve for the smallest integer n such that 3n + 5 > 26. Answer with just the integer.",
        "expected": "8",    # Because 3n > 21 => n > 7, smallest integer is 8
    },
    {
        "id": "commonsense_ice",
        "type": "text",
        "prompt": (
            "You place an ice cube in a glass of water and mark the water level. "
            "After the ice melts, does the water level rise, fall, or stay the same? "
            "Answer with exactly one of: 'rise', 'fall', 'stay the same'."
        ),
        "expected": "stay the same",
    },
    {
        "id": "logic_race",
        "type": "text",
        "prompt": (
            "In a race, you pass the person in second place. What position are you now in? "
            "Answer with a single word like 'first', 'second', 'third'."
        ),
        "expected": "second",
    },
    {
        "id": "logic_race",
        "prompt": "Do flying fish have good eyesight?",
        "expected": 'true',
        "type": "common_sense"
    },
    {
        "id": "logic_race",
        "prompt": "If someone is a vegan, would they eat honey?",
        "expected": 'false',
        "type": "common_sense"
    },
    {
        "id": "logic_race",
        "prompt": "I am playing with a set of objects. Here are the actions I can do\n\n   Attack object\n   Feast object from another object\n   Succumb object\n   Overcome object from another object\n\nI have the following restrictions on my actions:\n    To perform Attack action, the following facts need to be true: Province object, Planet object, Harmony.\n    Once Attack action is performed the following facts will be true: Pain object.\n    Once Attack action is performed the following facts will be false: Province object, Planet object, Harmony.\n    To perform Succumb action, the following facts need to be true: Pain object.\n    Once Succumb action is performed the following facts will be true: Province object, Planet object, Harmony.    \n    Once Succumb action is performed the following facts will be false: Pain object.\n    To perform Overcome action, the following needs to be true: Province other object, Pain object.\n    Once Overcome action is performed the following will be true: Harmony, Province object, Object Craves other object.\n    Once Overcome action is performed the following will be false: Province other object, Pain object.\n    To perform Feast action, the following needs to be true: Object Craves other object, Province object, Harmony.\n    Once Feast action is performed the following will be true: Pain object, Province other object.\n    Once Feast action is performed the following will be false:, Object Craves other object, Province object, Harmony.\n\n[STATEMENT]\nAs initial conditions I have that, object b craves object d, object c craves object b, harmony, planet object a, planet object d, province object a and province object c.\nMy goal is to have that object b craves object c and object d craves object b.\n\nMy plan is as follows:\n\n[PLAN]\nfeast object c from object b\nsuccumb object c\nfeast object b from object d\novercome object b from object c\nattack object d\novercome object d from object b\n[PLAN END]\n\n[STATEMENT]\nAs initial conditions I have that, object a craves object d, object b craves object a, object d craves object c, harmony, planet object c and province object b.\nMy goal is to have that object a craves object d and object c craves object b.\n\nMy plan is as follows:\n\n[PLAN]",
        "expected": "(feast b a)\n(succumb b)\n(feast a d)\n(overcome a b)\n(feast d c)\n(succumb d)\n(feast a b)\n(overcome a d)\n(attack c)\n(overcome c b)\n",
        "type": "planning"
    },
    {
        "id": "logic_race",
        "prompt": "I have to plan logistics to transport packages within cities via trucks and between cities via airplanes. Locations within a city are directly connected (trucks can move between any two such locations), and so are the cities. In each city there is exactly one truck and each city has one location that serves as an airport.\nHere are the actions that can be performed:\n\nLoad a package into a truck. For example, load package_1 into truck_1 at location_1_1.\nLoad a package into an airplane. For example, load package_1 into airplane_1 at location_1_1.\nUnload a package from a truck. For example, unload package_1 from truck_1 at location_1_1.\nUnload a package from an airplane. For example, unload package_1 from airplane_1 at location_1_1.\nDrive a truck from one location to another location. For example, drive truck_1 from location_1_1 to location_1_2 in city_1.\nFly an airplane from one city to another city. For example, fly airplane_1 from location_1_1 to location_2_1. Here location_1_1 is the airport in city_1 and location_2_1 is the airport in city_2.\n\nThe following are the restrictions on the actions:\nA package can be loaded into a truck only if the package and the truck are in the same location.\nOnce a package is loaded into a truck, the package is not at the location and is in the truck.   \nA package can be loaded into an airplane only if the package and the airplane are in the same location.\nOnce a package is loaded into an airplane, the package is not at the location and is in the airplane.\nA package can be unloaded from a truck only if the package is in the truck.\nOnce a package is unloaded from a truck, the package is not in the truck and is at the location of the truck.\nA package can be unloaded from an airplane only if the package in the airplane.\nOnce a package is unloaded from an airplane, the package is not in the airplane and is at the location of the airplane.   \nA truck can be driven from one location to another if the truck is at the from-location and both from-location and to-location are locations in the same city.\nOnce a truck is driven from one location to another, it is not at the from-location and is at the to-location.\nAn airplane can be flown from one city to another if the from-location and the to-location are airports and the airplane is at the from-location.\nOnce an airplane is flown from one city to another the airplane is not at the from-location and is at the to-location.\n\n[STATEMENT]\nAs initial conditions I have that, location_0_0 is an airport, location_1_0 is an airport, location_2_0 is an airport, airplane_0 is at location_0_0, package_0 is at location_0_2, package_1 is at location_1_1, package_2 is at location_2_1, truck_0 is at location_0_0, truck_1 is at location_1_0, truck_2 is at location_2_0, location_0_0 is in the city city_0, location_0_1 is in the city city_0, location_0_2 is in the city city_0, location_1_0 is in the city city_1, location_1_1 is in the city city_1, location_1_2 is in the city city_1, location_2_0 is in the city city_2, location_2_1 is in the city city_2 and location_2_2 is in the city city_2.\nMy goal is to have that package_0 is at location_0_2, package_1 is at location_1_2 and package_2 is at location_2_2.\n\nMy plan is as follows:\n\n[PLAN]\ndrive truck_2 from location_2_0 to location_2_1 in city_2\nload package_2 into truck_2 at location_2_1\ndrive truck_2 from location_2_1 to location_2_2 in city_2\nunload package_2 from truck_2 at location_2_2\ndrive truck_1 from location_1_0 to location_1_1 in city_1\nload package_1 into truck_1 at location_1_1\ndrive truck_1 from location_1_1 to location_1_2 in city_1\nunload package_1 from truck_1 at location_1_2\n[PLAN END]\n\n[STATEMENT]\nAs initial conditions I have that, location_0_0 is an airport, location_1_0 is an airport, location_2_0 is an airport, airplane_0 is at location_1_0, package_0 is at location_0_1, package_1 is at location_1_1, package_2 is at location_0_1, truck_0 is at location_0_1, truck_1 is at location_1_0, truck_2 is at location_2_1, location_0_0 is in the city city_0, location_0_1 is in the city city_0, location_0_2 is in the city city_0, location_1_0 is in the city city_1, location_1_1 is in the city city_1, location_1_2 is in the city city_1, location_2_0 is in the city city_2, location_2_1 is in the city city_2 and location_2_2 is in the city city_2.\nMy goal is to have that package_0 is at location_2_0, package_1 is at location_2_2 and package_2 is at location_2_2.\n\nMy plan is as follows:\n\n[PLAN]",
        "expected": "(drive-truck t2 l2-1 l2-0 c2)\n(load-truck p2 t0 l0-1)\n(load-truck p0 t0 l0-1)\n(drive-truck t0 l0-1 l0-0 c0)\n(unload-truck p2 t0 l0-0)\n(unload-truck p0 t0 l0-0)\n(drive-truck t1 l1-0 l1-1 c1)\n(load-truck p1 t1 l1-1)\n(drive-truck t1 l1-1 l1-0 c1)\n(unload-truck p1 t1 l1-0)\n(load-airplane p1 a0 l1-0)\n(fly-airplane a0 l1-0 l0-0)\n(load-airplane p2 a0 l0-0)\n(load-airplane p0 a0 l0-0)\n(fly-airplane a0 l0-0 l2-0)\n(unload-airplane p2 a0 l2-0)\n(load-truck p2 t2 l2-0)\n(unload-airplane p1 a0 l2-0)\n(load-truck p1 t2 l2-0)\n(drive-truck t2 l2-0 l2-2 c2)\n(unload-truck p2 t2 l2-2)\n(unload-truck p1 t2 l2-2)\n(unload-airplane p0 a0 l2-0)\n",
        "type": "planning"
    },
    {
        "id": "logic_race",
        "prompt": "I am playing with a set of objects. Here are the actions I can do\n\nPaltry object_0 object_1 object_2.\nSip object_0 object_1 object_2.\nClip object_0 object_1 object_2.\nWretched object_0 object_1 object_2 object_3.\nMemory object_0 object_1 object_2.\nTightfisted object_0 object_1 object_2.\n\nI have the following restrictions on my actions:\nTo perform paltry action, the following facts need to be true: hand object_0, cats object_1, texture object_2, vase object_0 object_1, and next object_1 object_2\nOnce paltry is performed the following facts will be true: next object_0 object_2\nOnce paltry is performed the following facts will be false: vase object_0 object_1\nTo perform sip action, the following facts need to be true: hand object_0, cats object_1, texture object_2, next object_0 object_2, and next object_1 object_2\nOnce sip is performed the following facts will be true: vase object_0 object_1\nOnce sip is performed the following facts will be false: next object_0 object_2\nTo perform clip action, the following facts need to be true: hand object_0, sneeze object_1, texture object_2, next object_1 object_2, and next object_0 object_2\nOnce clip is performed the following facts will be true: vase object_0 object_1\nOnce clip is performed the following facts will be false: next object_0 object_2\nTo perform wretched action, the following facts need to be true: sneeze object_0, texture object_1, texture object_2, stupendous object_3, next object_0 object_1, collect object_1 object_3, and collect object_2 object_3\nOnce wretched is performed the following facts will be true: next object_0 object_2\nOnce wretched is performed the following facts will be false: next object_0 object_1\nTo perform memory action, the following facts need to be true: cats object_0, spring object_1, spring object_2, and next object_0 object_1\nOnce memory is performed the following facts will be true: next object_0 object_2\nOnce memory is performed the following facts will be false: next object_0 object_1\nTo perform tightfisted action, the following facts need to be true: hand object_0, sneeze object_1, texture object_2, next object_1 object_2, and vase object_0 object_1\nOnce tightfisted is performed the following facts will be true: next object_0 object_2\nOnce tightfisted is performed the following facts will be false: vase object_0 object_1\n\n[STATEMENT]\nAs initial conditions I have that, cats object_0, cats object_1, collect object_10 object_3, collect object_11 object_3, collect object_12 object_4, collect object_13 object_4, collect object_8 object_2, collect object_9 object_2, hand object_14, hand object_15, hand object_16, hand object_17, hand object_18, next object_0 object_12, next object_1 object_8, next object_14 object_10, next object_15 object_11, next object_16 object_11, next object_17 object_13, next object_18 object_12, next object_5 object_9, next object_6 object_11, next object_7 object_12, sneeze object_5, sneeze object_6, sneeze object_7, spring object_10, spring object_12, spring object_8, stupendous object_2, stupendous object_3, stupendous object_4, texture object_10, texture object_11, texture object_12, texture object_13, texture object_8 and texture object_9.\nMy goal is to have that next object_14 object_11, next object_15 object_9, next object_16 object_10, next object_17 object_10 and next object_18 object_10.\n\nMy plan is as follows:\n\n[PLAN]\nclip object_16 object_6 object_11\nclip object_15 object_6 object_11\nsip object_18 object_0 object_12\nwretched object_7 object_12 object_13 object_4\nclip object_17 object_7 object_13\nwretched object_7 object_13 object_12 object_4\ntightfisted object_17 object_7 object_12\nsip object_17 object_0 object_12\nwretched object_6 object_11 object_10 object_3\ntightfisted object_16 object_6 object_10\nclip object_14 object_6 object_10\nmemory object_0 object_12 object_10\npaltry object_18 object_0 object_10\npaltry object_17 object_0 object_10\ntightfisted object_15 object_6 object_10\nwretched object_6 object_10 object_11 object_3\ntightfisted object_14 object_6 object_11\nsip object_15 object_0 object_10\nmemory object_0 object_10 object_8\npaltry object_15 object_0 object_8\nwretched object_5 object_9 object_8 object_2\nclip object_15 object_5 object_8\nwretched object_5 object_8 object_9 object_2\ntightfisted object_15 object_5 object_9\n[PLAN END]\n\n[STATEMENT]\nAs initial conditions I have that, cats object_0, cats object_1, collect object_10 object_3, collect object_11 object_3, collect object_12 object_4, collect object_13 object_4, collect object_8 object_2, collect object_9 object_2, hand object_14, hand object_15, hand object_16, hand object_17, hand object_18, next object_0 object_10, next object_1 object_12, next object_14 object_8, next object_15 object_11, next object_16 object_13, next object_17 object_9, next object_18 object_13, next object_5 object_8, next object_6 object_10, next object_7 object_12, sneeze object_5, sneeze object_6, sneeze object_7, spring object_10, spring object_12, spring object_8, stupendous object_2, stupendous object_3, stupendous object_4, texture object_10, texture object_11, texture object_12, texture object_13, texture object_8 and texture object_9.\nMy goal is to have that next object_14 object_11, next object_15 object_12, next object_16 object_8, next object_17 object_9 and next object_18 object_13.\n\nMy plan is as follows:\n\n[PLAN]",
        "expected": "(wretched o7 o12 o13 o4)\n(clip o16 o7 o13)\n(wretched o7 o13 o12 o4)\n(tightfisted o16 o7 o12)\n(sip o16 o1 o12)\n(memory o1 o12 o8)\n(paltry o16 o1 o8)\n(sip o14 o1 o8)\n(memory o1 o8 o10)\n(paltry o14 o1 o10)\n(clip o14 o6 o10)\n(wretched o6 o10 o11 o3)\n(clip o15 o6 o11)\n(tightfisted o14 o6 o11)\n(wretched o6 o11 o10 o3)\n(tightfisted o15 o6 o10)\n(sip o15 o1 o10)\n(memory o1 o10 o12)\n(paltry o15 o1 o12)\n",
        "type": "planning"
    }
]


In [ ]:
# Tree of thought (X of thought)
# Reasoning via planning
# 'Wait' am i correct? Double check
# Critic 
# Send to output

# Future:
# Implement RAG or memory of some kind to grab from text data

In [3]:
def reasoning_via_planning(question: str):
    SYSTEM_PROMPT = "You are a helpful assistant that reasons via step by step planning.",
    reasoning_str = "You will decompose the problem down into a step by step plan to solve the question provided. Carefully consider each step before moving on to the next. Once you have a plan, execute each step in order to arrive at the final answer. Be sure to double-check your work at each step to ensure accuracy.\n\n Question: "
    return call_model_chat_completions(reasoning_str + question, model=MODEL)

In [ ]:

# Example:
results_llm_judge = self_evaluate_tests(tests, verbose=True, model=MODEL, grader_model=MODEL)